# 01 · Data Preprocessing

Replicates `step1_importDataToMatlab.m` + `step2_preprocessData.m`.

For each dataset this notebook:
1. Reads the raw CSV
2. Applies the three exclusion criteria from the paper
3. Applies dataset-specific confidence transformations
4. Saves a per-dataset `.npz` file for use in notebook 02

**Datasets processed**
| Dataset | Raw subjects | After filtering | Notes |
|---------|-------------|-----------------|-------|
| Haddara 2022 Expt2 | 75 | 70 | 4-point conf, 6 test days |
| Maniscalco 2017 Expt1 | 30 | ~22 | 4-point conf |
| Shekhar 2021 | 20 | 20 | continuous conf → 6 bins |
| Rouault 2018 Expt1 | 498 | ~466 | conf − 5 (clip to ≥ 1) |
| Rouault 2018 Expt2 | 497 | ~484 | 6-point conf |
| Locke 2020 | 10 | 10 | binary conf + 1, 7 conditions |


In [ ]:
import sys, os, warnings
warnings.filterwarnings('ignore')

# ── path resolution (works whether run from repo root or notebooks/ dir)
REPO = os.path.abspath(os.path.join(os.getcwd(),
    '..' if os.path.basename(os.getcwd()) == 'notebooks' else '.'))
sys.path.insert(0, os.path.join(REPO, 'src'))

import numpy as np
import pandas as pd
from metasignal.stdpy.compute_all import compute_all_measures

DATA = os.path.join(REPO, 'matlab', 'metasignal_mat', 'Preprocess', 'orig_csv_files')
OUT  = os.path.join(REPO, 'notebooks', 'precomputed')
os.makedirs(OUT, exist_ok=True)

# Exclusion thresholds (match step2_preprocessData.m)
ACC_LO, ACC_HI    = 0.60, 0.95
MAX_PROP_SAME     = 0.85

print(f'Data directory:   {DATA}')
print(f'Output directory: {OUT}')
print(f'Datasets present: {[f for f in os.listdir(DATA) if f.endswith(".csv")]}')

## Exclusion criteria (from step2_preprocessData.m)

A subject is excluded if any of the following hold:
- Accuracy < 0.60 or > 0.95
- Mode response proportion > 0.85 (response stereotypy)
- Mode confidence proportion > 0.85 (confidence stereotypy)

In [ ]:
def filter_subjects(grp_iter, conf_transform=None, extra_cols=None):
    """
    grp_iter: iterable of (sid, DataFrame) groups
    conf_transform: optional function(conf_array) -> conf_array
    extra_cols: list of additional column names to keep
    Returns list of subject dicts.
    """
    subjects = []
    excluded = 0
    for sid, grp in grp_iter:
        grp = grp.dropna(subset=['Stimulus', 'Response', 'Confidence'])
        if len(grp) == 0:
            excluded += 1
            continue
        stim = grp['Stimulus'].to_numpy(float)
        resp = grp['Response'].to_numpy(float)
        conf = grp['Confidence'].to_numpy(float)
        if conf_transform:
            conf = conf_transform(conf)

        acc  = np.mean(stim == resp)
        pr   = np.max(np.unique(resp, return_counts=True)[1]) / len(resp)
        pc   = np.max(np.unique(np.round(conf).astype(int), return_counts=True)[1]) / len(conf)

        if acc < ACC_LO or acc > ACC_HI or pr > MAX_PROP_SAME or pc > MAX_PROP_SAME:
            excluded += 1
            continue

        rec = {'sid': sid, 'stim': stim, 'resp': resp, 'conf': conf}
        if extra_cols:
            for col in extra_cols:
                rec[col] = grp[col].to_numpy(float)
        subjects.append(rec)
    return subjects, excluded

## Haddara 2022 (Expt 2)
4-point confidence scale, 7 testing days (we use days 1–7).

In [ ]:
df_ha = pd.read_csv(os.path.join(DATA, 'data_Haddara_2022_Expt2.csv'))
ha_subs, ha_excl = filter_subjects(
    df_ha.groupby('Subj_idx'),
    extra_cols=['Day']
)
for s in ha_subs:
    s['n_ratings'] = 4

np.savez(os.path.join(OUT, 'haddara.npz'), subjects=ha_subs)
print(f'Haddara: {len(ha_subs)} included, {ha_excl} excluded  (paper reports n=70)')
print(f'  Confidence range: {min(s["conf"].min() for s in ha_subs):.0f}–'
      f'{max(s["conf"].max() for s in ha_subs):.0f}')

## Maniscalco 2017 (Expt 1)
4-point confidence. Some trials have NaN responses (excluded per subject).

In [ ]:
df_ma = pd.read_csv(os.path.join(DATA, 'data_Maniscalco_2017_expt1.csv'))
ma_subs, ma_excl = filter_subjects(df_ma.groupby('Subj_idx'))
for s in ma_subs:
    s['n_ratings'] = 4

np.savez(os.path.join(OUT, 'maniscalco.npz'), subjects=ma_subs)
print(f'Maniscalco: {len(ma_subs)} included, {ma_excl} excluded  (paper reports n=22)')

## Shekhar 2021
Confidence is continuous (50–100%). Discretised into 6 equal bins matching the MATLAB analysis:
```python
edges = np.linspace(50, 100, 7)  # [50, 58.33, 66.67, 75, 83.33, 91.67, 100]
conf  = np.digitize(conf_raw, edges, right=True).clip(1, 6)
```
Contrast levels: 1 = hardest (67% acc), 2 = medium, 3 = easiest (89% acc).

In [ ]:
df_sh = pd.read_csv(os.path.join(DATA, 'data_Shekhar_2021.csv'))
N_RATINGS_SH = 6
edges_sh = np.linspace(50, 100, N_RATINGS_SH + 1)

def discretize_shekhar(conf_raw):
    return np.clip(np.digitize(conf_raw, edges_sh, right=True), 1, N_RATINGS_SH).astype(float)

sh_subs, sh_excl = filter_subjects(
    df_sh.groupby('Subj_idx'),
    conf_transform=discretize_shekhar,
    extra_cols=['Contrast']
)
for s in sh_subs:
    s['n_ratings'] = N_RATINGS_SH

np.savez(os.path.join(OUT, 'shekhar.npz'), subjects=sh_subs)
print(f'Shekhar: {len(sh_subs)} included, {sh_excl} excluded  (paper reports n=20)')
for c in [1, 2, 3]:
    accs = [np.mean(s['stim'][s['Contrast']==c] == s['resp'][s['Contrast']==c])
            for s in sh_subs if (s['Contrast']==c).any()]
    print(f'  Contrast {c}: mean accuracy = {np.mean(accs):.3f}')

## Rouault 2018 (Expt 1 & 2)

**Expt 1** — raw confidence is 1–11; MATLAB transforms with `conf = conf - 5; conf(conf<1) = 1`, giving a 1–6 scale.

**Expt 2** — confidence is already 1–6, no transformation needed.

Difficulty is manipulated via `DotDiff` (dot difference); larger values = easier.

In [ ]:
# Rouault Expt 1
df_r1 = pd.read_csv(os.path.join(DATA, 'data_Rouault_2018_Expt1.csv'))
r1_subs, r1_excl = filter_subjects(
    df_r1.groupby('Subj_idx'),
    conf_transform=lambda c: np.clip(c - 5, 1, None),
    extra_cols=['DotDiff']
)
for s in r1_subs:
    s['n_ratings'] = 6
    s['contrast'] = s.pop('DotDiff')

np.savez(os.path.join(OUT, 'rouault1.npz'), subjects=r1_subs)
print(f'Rouault1: {len(r1_subs)} included, {r1_excl} excluded  (paper reports n=466)')

# Rouault Expt 2
df_r2 = pd.read_csv(os.path.join(DATA, 'data_Rouault_2018_Expt2.csv'))
r2_subs, r2_excl = filter_subjects(
    df_r2.groupby('Subj_idx'),
    extra_cols=['DotDiff']
)
for s in r2_subs:
    s['n_ratings'] = 6
    s['contrast'] = s.pop('DotDiff')

np.savez(os.path.join(OUT, 'rouault2.npz'), subjects=r2_subs)
print(f'Rouault2: {len(r2_subs)} included, {r2_excl} excluded  (paper reports n=484)')

## Locke 2020
Binary confidence (0/1 → recoded to 1/2). 7 conditions with different response biases.  
Training trials (Training==1) are excluded.

In [ ]:
df_lo = pd.read_csv(os.path.join(DATA, 'data_Locke_2020.csv'))
df_lo = df_lo[df_lo['Training'] == 0].copy()
df_lo['Confidence'] = df_lo['Confidence'] + 1  # 0/1 → 1/2

lo_subs, lo_excl = filter_subjects(
    df_lo.groupby('Subj_idx'),
    extra_cols=['Condition']
)
for s in lo_subs:
    s['n_ratings'] = 2
    s['condition'] = s.pop('Condition')

np.savez(os.path.join(OUT, 'locke.npz'), subjects=lo_subs)
print(f'Locke: {len(lo_subs)} included, {lo_excl} excluded  (paper reports n=10)')
print(f'  Conditions: {sorted(set(c for s in lo_subs for c in np.unique(s["condition"]).tolist()))}')

## Summary

In [ ]:
import pandas as pd
summary = pd.DataFrame([
    {'Dataset': 'Haddara 2022',     'n (Python)': len(ha_subs), 'n (paper)': 70,  'n_ratings': 4},
    {'Dataset': 'Maniscalco 2017', 'n (Python)': len(ma_subs), 'n (paper)': 22,  'n_ratings': 4},
    {'Dataset': 'Shekhar 2021',    'n (Python)': len(sh_subs), 'n (paper)': 20,  'n_ratings': 6},
    {'Dataset': 'Rouault 2018 E1', 'n (Python)': len(r1_subs), 'n (paper)': 466, 'n_ratings': 6},
    {'Dataset': 'Rouault 2018 E2', 'n (Python)': len(r2_subs), 'n (paper)': 484, 'n_ratings': 6},
    {'Dataset': 'Locke 2020',      'n (Python)': len(lo_subs), 'n (paper)': 10,  'n_ratings': 2},
])
print(summary.to_string(index=False))
print('\nAll processed datasets saved to notebooks/precomputed/')